# Channel template: 从声明到 ODE

这个 notebook 解释当前 `braincell` 中 channel template 的核心思想：把通道的门控变量、稳态函数、时间常数或转移速率写成类方法，然后由 template 自动生成对应的 ODE。

这里聚焦两个当前源码里的通用模板：

- `braincell.channel.HH`：Hodgkin-Huxley 门控变量模板。
- `braincell.channel.Markov`：概率状态转移模板。

本 notebook 主要展开 `HH`，因为目前仓库里大量 sodium、potassium、calcium channel 都使用这个模板。

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import brainunit as u
import jax.numpy as jnp

import braincell
from braincell import IonInfo
from braincell.channel import NaF_SU2015_DCN, Na_HH1952
from braincell.channel._base import Gate, HH, Markov, Transition

print("braincell version:", braincell.__version__)

## 1. `HH` template 的数学形式

`HH` template 使用 `gates` 声明门控变量。每个 `Gate` 至少有：

- `name`：状态名，例如 `m`、`h`、`p`。
- `power`：电导因子里的指数，例如 `m^3 h` 中 `m` 的 `power=3`。
- `phi` 或 `q10/temp_ref`：可选温度修正因子。

对每个 gate，template 自动识别两类写法之一。

第一类是 `inf/tau`：

$$
\frac{dx}{dt} = \phi_x \frac{x_\infty(V, \mathrm{ions}) - x}{\tau_x(V, \mathrm{ions})}\;\frac{1}{\mathrm{ms}}
$$

对应代码方法：`f_x_inf(...)` 和 `f_x_tau(...)`。

第二类是 `alpha/beta`：

$$
\frac{dx}{dt} = \phi_x \left(\alpha_x(V, \mathrm{ions})(1-x) - \beta_x(V, \mathrm{ions})x\right)\;\frac{1}{\mathrm{ms}}
$$

对应代码方法：`f_x_alpha(...)` 和 `f_x_beta(...)`。

注意当前约定：`f_*_tau`、`f_*_alpha`、`f_*_beta` 返回的是以 ms 或 1/ms 为自然单位的数值，`HH.compute_derivative()` 统一除以 `u.ms`，所以得到的 derivative 带有 `1/ms` 量纲。

In [ ]:
print("NaF_SU2015_DCN gates:", NaF_SU2015_DCN.gates)
print("Na_HH1952 gates:", Na_HH1952.gates)

## 2. HH gate 的温度矫正因子

`HH` template 在每个 gate 上用 `gate_phi(gate)` 得到温度矫正因子。解析顺序是：

1. 如果 `Gate(..., phi=...)` 显式给了 `phi`，直接使用它。
2. 如果给了 `q10` 和 `temp_ref`，使用当前 channel 的 `self.temp` 计算：

$$
\phi = q_{10}^{(T - T_{ref}) / 10\mathrm{K}}
$$

3. 如果都没有，默认 `phi = 1`。

因此 `inf/tau` gate 的真实 ODE 是：

$$
\frac{dx}{dt} = \phi_x \frac{x_\infty - x}{\tau_x}\frac{1}{\mathrm{ms}}
$$

`alpha/beta` gate 也同样乘以 `phi_x`。

In [ ]:
default_temp = Na_HH1952(size=1)
warmer_than_ref = Na_HH1952(
    size=1,
    temp=u.celsius2kelvin(36.0),
    temp_ref=u.celsius2kelvin(26.0),
    q10=3.0,
)

gate = Na_HH1952.gates[0]

print("default phi:", default_temp.gate_phi(gate))
print("q10 corrected phi:", warmer_than_ref.gate_phi(gate))

## 3. 仓库例子：`NaF_SU2015_DCN` 的 `inf/tau` 写法

`NaF_SU2015_DCN` 来自 `braincell/channel/sodium.py`，它声明：

```python
gates = (
    Gate("m", power=3),
    Gate("h"),
)
```

因此 conductance factor 是：

$$
m^3h
$$

电流公式写在 `current()` 里：

$$
I_{Na} = g_{max}m^3h(E_{Na} - V)
$$

它同时提供 `f_m_inf`、`f_m_tau`、`f_h_inf`、`f_h_tau`，所以 template 自动生成：

$$
\frac{dm}{dt}=\frac{m_\infty(V)-m}{\tau_m(V)}\frac{1}{\mathrm{ms}},\qquad
\frac{dh}{dt}=\frac{h_\infty(V)-h}{\tau_h(V)}\frac{1}{\mathrm{ms}}
$$

In [ ]:
na = IonInfo(
    Ci=jnp.array([10.0]) * u.mM,
    Co=jnp.array([140.0]) * u.mM,
    E=jnp.array([50.0]) * u.mV,
    valence=1,
)
V = jnp.array([-55.0]) * u.mV

ch = NaF_SU2015_DCN(size=1)
ch.init_state(V, na)
ch.reset_state(V, na)

print("m reset to m_inf:", ch.m.value)
print("h reset to h_inf:", ch.h.value)
print("m_tau:", ch.f_m_tau(V, na))
print("h_tau:", ch.f_h_tau(V, na))
print("conductance factor m^3 h:", ch.conductance_factor(V, na))
print("current:", ch.current(V, na).to_decimal(u.mA / u.cm**2), "mA/cm^2")

上一个 cell 中 `reset_state()` 会把门控变量设置到稳态，因此 `compute_derivative()` 后 derivative 接近 0。为了直接看到公式，我们手动把 `m` 和 `h` 偏离稳态，再比较 template 计算结果和手写公式。

In [ ]:
ch.m.value = jnp.array([0.1])
ch.h.value = jnp.array([0.8])

ch.compute_derivative(V, na)

expected_dm = (ch.f_m_inf(V, na) - ch.m.value) / ch.f_m_tau(V, na) / u.ms
expected_dh = (ch.f_h_inf(V, na) - ch.h.value) / ch.f_h_tau(V, na) / u.ms

print("dm/dt from template:", ch.m.derivative.to_decimal(u.Hz), "Hz")
print("dm/dt by formula:   ", expected_dm.to_decimal(u.Hz), "Hz")
print("dh/dt from template:", ch.h.derivative.to_decimal(u.Hz), "Hz")
print("dh/dt by formula:   ", expected_dh.to_decimal(u.Hz), "Hz")

## 4. 一个最小自定义 `HH` channel

如果一个新通道可以写成 HH gate 形式，通常只需要：

1. 继承 `HH`。
2. 声明 `root_type`，说明它绑定哪类 ion。
3. 声明 `gates`。
4. 实现每个 gate 的 `inf/tau` 或 `alpha/beta` 方法。
5. 实现 `current()`。

In [ ]:
class DemoKChannel(HH):
    root_type = braincell.ion.Potassium
    gates = (Gate("n", power=4, phi=2.0),)

    def __init__(self, size, g_max=1.0 * (u.mS / u.cm**2)):
        super().__init__(size=size)
        self.g_max = g_max

    def current(self, V, K: IonInfo):
        return self.g_max * self.conductance_factor(V, K) * (K.E - V)

    def f_n_inf(self, V, K: IonInfo):
        _ = K
        V_mV = V.to_decimal(u.mV)
        return 1.0 / (1.0 + u.math.exp(-(V_mV + 35.0) / 10.0))

    def f_n_tau(self, V, K: IonInfo):
        _ = (V, K)
        return 5.0


k = IonInfo(
    Ci=jnp.array([54.4]) * u.mM,
    Co=jnp.array([2.5]) * u.mM,
    E=jnp.array([-90.0]) * u.mV,
    valence=1,
)
demo = DemoKChannel(size=1)
demo.init_state(V, k)
demo.n.value = jnp.array([0.2])
demo.compute_derivative(V, k)

print("n_inf:", demo.f_n_inf(V, k))
print("dn/dt:", demo.n.derivative.to_decimal(u.Hz), "Hz")
print("current:", demo.current(V, k).to_decimal(u.mA / u.cm**2), "mA/cm^2")

## 5. `alpha/beta` 写法：`Na_HH1952`

`Na_HH1952` 使用 `p^3q` 形式，但 gate 方法不是 `inf/tau`，而是 `alpha/beta`：

$$
\frac{dp}{dt}=\phi(\alpha_p(1-p)-\beta_p p)\frac{1}{\mathrm{ms}}
$$

`reset_state()` 对这类 gate 使用稳态值：

$$
p_\infty = \frac{\alpha_p}{\alpha_p + \beta_p}
$$

In [ ]:
hh_na = Na_HH1952(size=1)
hh_na.init_state(V, na)
hh_na.reset_state(V, na)

p_inf = hh_na.f_p_alpha(V, na) / (hh_na.f_p_alpha(V, na) + hh_na.f_p_beta(V, na))
q_inf = hh_na.f_q_alpha(V, na) / (hh_na.f_q_alpha(V, na) + hh_na.f_q_beta(V, na))

print("p reset:", hh_na.p.value)
print("p alpha/(alpha+beta):", p_inf)
print("q reset:", hh_na.q.value)
print("q alpha/(alpha+beta):", q_inf)

## 6. `Markov` template 的对应关系

`Markov` template 用 `Transition(src, dst, forward, backward)` 表示概率状态之间的流量。

如果有一个双向转移：

$$
C \xrightleftharpoons[\beta]{\alpha} O
$$

则 ODE 是：

$$
\frac{dC}{dt}= -C\alpha + O\beta,\qquad
\frac{dO}{dt}= C\alpha - O\beta
$$

`Markov` 会为独立状态创建 `DiffEqState`，并用守恒量重建一个 dependent state，适合多状态概率通道。它和 `HH` 的区别是：`HH` 直接声明每个 gate 的一阶动力学；`Markov` 声明状态图和转移率。

从数学上看，HH gate 可以理解为多个独立的 2-state Markov gate。比如 `m^3h` 表示 3 个独立激活 gate 和 1 个失活 gate 都满足各自的两状态动力学，最后开放概率写成乘积：

$$
P_{open}^{HH} = m^3h
$$

也就是说，HH 的多个 gate 是相互独立的，只有所有所需 gate 都处在开放构象时，channel 才开放。

通常说的 Markov channel 更常指 3 个及以上状态的状态图。状态之间不再拆成独立 gate，而是直接描述整个 channel 分子处于 closed、open、inactivated 等构象的概率。开放概率也不再是 gate 的乘积，而是所有 open 相关状态概率的加和：

$$
P_{open}^{Markov}=\sum_{s\in OpenStates} P_s
$$

下面给出一个完整可运行的三状态 potassium Markov channel。我们只保存 `O` 和 `I`，把 `C` 作为 dependent state 用守恒量重建：

$$
C + O + I = 1
$$

如果直接写状态方程，`C`、`O`、`I` 三个概率各有一个 ODE，本来是 3 维系统。但它们不是独立变量，因为总概率守恒给了一个额外约束。把这个约束带入方程后，可以消掉一个状态，实际只需要积分 2 个独立状态。

在 `braincell.channel.Markov` 里，被消掉、运行时通过守恒条件重建的状态就叫 `dependent_state`。下面例子设置 `dependent_state = "C"`，所以：

$$
C = 1 - O - I
$$

转移图是：

$$
C \xrightleftharpoons[\beta]{\alpha} O \xrightleftharpoons[\delta]{\gamma} I
$$

因此独立状态的 ODE 是：

$$
\frac{dO}{dt} = (C\alpha - O\beta - O\gamma + I\delta)\frac{1}{\mathrm{ms}}
$$

$$
\frac{dI}{dt} = (O\gamma - I\delta)\frac{1}{\mathrm{ms}}
$$

这个例子只有一个 open state，所以 `P_open = O`。如果一个模型有多个 open state，例如 `O1` 和 `O2`，电流里就应使用 `O1 + O2`。

In [ ]:
class DemoKMarkov3State(Markov):
    root_type = braincell.ion.Potassium
    pairs = (
        Transition("C", "O", "alpha", "beta"),
        Transition("O", "I", "gamma", "delta"),
    )
    dependent_state = "C"
    conserve = 1.0

    def __init__(self, size, g_max=1.0 * (u.mS / u.cm**2)):
        super().__init__(size=size)
        self.g_max = g_max

    def alpha(self, V, K: IonInfo):
        _ = (V, K)
        return 0.2

    def beta(self, V, K: IonInfo):
        _ = (V, K)
        return 0.1

    def gamma(self, V, K: IonInfo):
        _ = (V, K)
        return 0.05

    def delta(self, V, K: IonInfo):
        _ = (V, K)
        return 0.02

    def current(self, V, K: IonInfo):
        states = self.state_values()
        open_probability = states["O"]
        return self.g_max * open_probability * (K.E - V)


mk = DemoKMarkov3State(size=1)
mk.init_state(V, k)
mk.O.value = jnp.array([0.25])
mk.I.value = jnp.array([0.10])
mk.compute_derivative(V, k)

states = mk.state_values()
expected_dO = (
    states["C"] * mk.alpha(V, k)
    - states["O"] * mk.beta(V, k)
    - states["O"] * mk.gamma(V, k)
    + states["I"] * mk.delta(V, k)
) / u.ms
expected_dI = (states["O"] * mk.gamma(V, k) - states["I"] * mk.delta(V, k)) / u.ms
open_probability = states["O"]

print("independent states:", mk.state_names)
print("dependent state:", mk.redundant_state)
print("state values:", states)
print("open probability:", open_probability)
print("dO/dt from template:", mk.O.derivative.to_decimal(u.Hz), "Hz")
print("dO/dt by formula:   ", expected_dO.to_decimal(u.Hz), "Hz")
print("dI/dt from template:", mk.I.derivative.to_decimal(u.Hz), "Hz")
print("dI/dt by formula:   ", expected_dI.to_decimal(u.Hz), "Hz")
print("current:", mk.current(V, k).to_decimal(u.mA / u.cm**2), "mA/cm^2")